# Daily H=22, notebook 3/3: rolling neural, explicabilidade e avaliação

Este notebook ajusta TSMixerX e LSTM nas 16 janelas 8/2/1 e depois executa
explicabilidade e avaliação final. Cada combinação modelo/janela salva pesos,
forecast, inputs, ranking e histórico.

**Tempo estimado no Colab com GPU:** 20–40 minutos. O teto operacional é 60
minutos, deixando 10 minutos para manifesto, ZIP e download antes do limite de
70 minutos.

In [ ]:
# Configuração compartilhada: repositório + armazenamento persistente
from pathlib import Path
import os, shutil, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('(Not on Colab / Drive mount skipped):', exc)

REPO_URL = 'https://github.com/hugogobato/Mestrado_Anna_Julia.git'
REPO = Path('/content/Mestrado_Anna_Julia')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

EXP = REPO / 'experiments/daily_h22'
PERSIST = Path('/content/drive/MyDrive/Mestrado_Anna_Julia/daily_h22')
DATA_DIR = PERSIST / 'data'
RESULTS_DIR = PERSIST / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ['DAILY_H22_DATA_DIR'] = str(DATA_DIR)
os.environ['DAILY_H22_RESULTS_DIR'] = str(RESULTS_DIR)
os.environ['MPLCONFIGDIR'] = '/content/matplotlib_cache'
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(EXP / 'requirements-colab.txt')], check=True)

def run_script(name, *args):
    command = [sys.executable, str(EXP / 'src' / name), *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, check=True, env=os.environ.copy())

print('Experiment:', EXP)
print('Persistent data:', DATA_DIR)
print('Persistent results:', RESULTS_DIR)
try:
    import torch
    print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
          '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('Torch check failed:', exc)


## Configuração da sessão

In [ ]:
RUN_MODE = 'final'  # 'smoke', 'validation' ou 'final'
SESSION_BUDGET_MINUTES = 60
MAX_STEPS = {'smoke': 30, 'validation': 200, 'final': 400}[RUN_MODE]
MAX_WINDOWS = {'smoke': 1, 'validation': 2, 'final': None}[RUN_MODE]
print(RUN_MODE, MAX_STEPS, MAX_WINDOWS)


## Ajustes rolling

O preset final executa as 16 janelas de ambos os modelos em uma única sessão.
O tempo é verificado entre ajustes e o script preserva margem para avaliação e
empacotamento.

In [ ]:
args = [
    '--models', 'tsmixerx', 'lstm',
    '--max-steps', MAX_STEPS,
    '--time-budget-minutes', SESSION_BUDGET_MINUTES,
]
if MAX_WINDOWS is not None:
    args += ['--max-windows', MAX_WINDOWS]
run_script('13_neural_rolling.py', *args)


## Cobertura

A avaliação final exige 16 janelas para cada modelo neural e para os
benchmarks do notebook 1.

In [ ]:
import pandas as pd
coverage = []
for name in ['tsmixerx', 'lstm']:
    path = RESULTS_DIR / f'forecasts/{name}.parquet'
    if path.exists():
        frame = pd.read_parquet(path)
        coverage.append({'model': name, 'windows': frame.window_id.nunique(), 'rows': len(frame)})
    else:
        coverage.append({'model': name, 'windows': 0, 'rows': 0})
coverage_df = pd.DataFrame(coverage)
display(coverage_df)
NEURAL_COMPLETE = bool((coverage_df.windows == 16).all())
print('Neural complete:', NEURAL_COMPLETE)
if RUN_MODE == 'final' and not NEURAL_COMPLETE:
    raise RuntimeError('Cobertura neural incompleta: o teto de 60 minutos foi atingido inesperadamente.')


## Explicabilidade temporal

Esta etapa roda automaticamente apenas quando os modelos neurais estão
completos. Integrated Gradients cobre quatro janelas e quatro horizontes.
Shapley Value Sampling gera valores de Shapley agrupados por feature e por lag
na janela final, reduzindo o custo.

In [ ]:
if NEURAL_COMPLETE:
    run_script('14_explainability.py', '--window-ids', 0, 5, 10, 15,
               '--origins-per-window', 3, '--horizons', 1, 5, 10, 22,
               '--n-steps', 32)
    run_script('14_explainability.py', '--window-ids', 15,
               '--origins-per-window', 1, '--horizons', 1, 22,
               '--n-steps', 16, '--run-shapley', '--shapley-samples', 10)
else:
    print('Explainability postponed until all neural windows are available.')


## Avaliação final

O script calcula métricas por `h=1,...,22`, DM e GW com HAC de 21 lags, MCS
em `h=22` com blocos de 22 pregões, gráficos VIX/RV e curvas de convergência.

In [ ]:
if NEURAL_COMPLETE:
    run_script('15_evaluate_daily.py', '--require-complete', '--mcs-reps', 1000)
    display(pd.read_csv(RESULTS_DIR / 'metrics/metrics_h22.csv').sort_values('MSE'))
    display(pd.read_csv(RESULTS_DIR / 'metrics/model_coverage.csv'))
else:
    print('Final evaluation postponed. Partial files remain safely stored in Drive.')


## Manifesto reproduzível

O manifesto registra versões, tamanhos e SHA-256 de todos os outputs existentes.

In [ ]:
run_script('16_package_results.py')
manifest = RESULTS_DIR / 'artifact_manifest.json'
if manifest.exists():
    print(manifest.read_text()[:4000])


## Verificação de recarga dos modelos

In [ ]:
artifacts = sorted((RESULTS_DIR / 'models/tsmixerx').glob('*.pt'))
print('TSMixerX artifacts:', len(artifacts))
if artifacts:
    sys.path.insert(0, str(EXP / 'src'))
    from models import DirectVolatilityRegressor
    loaded, payload = DirectVolatilityRegressor.load(artifacts[-1])
    print('Reloaded:', artifacts[-1].name, '| input_size:', loaded.input_size,
          '| features:', len(payload['features']))


In [ ]:
ARCHIVE_NAME = 'daily_h22_notebook3_results'
# Empacotamento e download seguro do estado atual
import shutil
from pathlib import Path

archive_base = Path('/content') / ARCHIVE_NAME
output_file = shutil.make_archive(str(archive_base), 'zip', root_dir=PERSIST)
print('Archive created:', output_file)
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
